In [ ]:
import pandas as pd
import numpy as np
import ast
from collections import Counter

# ============================================================
# ЗАГРУЗКА ДАННЫХ
# ============================================================
history = pd.read_parquet('artifacts/history_interactions.parquet')
games = pd.read_csv('data/game_details.csv')
users = pd.read_csv('data/unique_users.csv')
item_feat_mh = pd.read_parquet('artifacts/item_features_multihot.parquet')

mh_genre_cols = [c for c in item_feat_mh.columns if c.startswith('genres_')]
mh_cat_cols = [c for c in item_feat_mh.columns if c.startswith('categories_')]
mh_cols = mh_genre_cols + mh_cat_cols

print(f"Genre columns: {len(mh_genre_cols)}, Category columns: {len(mh_cat_cols)}")

# ============================================================
# A. ITEM POPULARITY FEATURES (из history)
# ============================================================
print("\n[A] Item popularity features...")

item_pop = history.groupby('appid').agg(
    item_n_players=('steamid', 'nunique'),
    item_total_playtime=('playtime_forever', 'sum'),
    item_avg_playtime=('playtime_forever', 'mean'),
    item_median_playtime=('playtime_forever', 'median'),
    item_std_playtime=('playtime_forever', 'std'),
    item_avg_target_hist=('target', 'mean'),
    item_std_target_hist=('target', 'std'),
    item_pct_high_target=('target', lambda x: (x >= 4).mean()),
    item_pct_zero_playtime=('playtime_forever', lambda x: (x == 0).mean()),
).reset_index()

item_pop['item_std_playtime'] = item_pop['item_std_playtime'].fillna(0)
item_pop['item_std_target_hist'] = item_pop['item_std_target_hist'].fillna(0)
item_pop['item_avg_playtime_log'] = np.log1p(item_pop['item_avg_playtime'])
item_pop['item_total_playtime_log'] = np.log1p(item_pop['item_total_playtime'])
item_pop['item_n_players_log'] = np.log1p(item_pop['item_n_players'])

print(f"  Items with popularity data: {len(item_pop)}")

# ============================================================
# B. DEVELOPER / PUBLISHER FEATURES
# ============================================================
print("\n[B] Developer/Publisher features...")

def parse_json_col(series):
    def _parse(x):
        if pd.isna(x) or x == '[]':
            return []
        try:
            return ast.literal_eval(str(x))
        except:
            return []
    return series.apply(_parse)

games['developers_list'] = parse_json_col(games['developers'])
games['publishers_list'] = parse_json_col(games['publishers'])

# --- Developer exploded ---
dev_exploded = games[['appid', 'developers_list']].explode('developers_list')
dev_exploded = dev_exploded.rename(columns={'developers_list': 'developer'})
dev_exploded = dev_exploded.dropna(subset=['developer'])

# Developer stats from history
dev_history = dev_exploded.merge(
    history[['appid', 'steamid', 'playtime_forever', 'target']], on='appid', how='inner'
)
dev_stats = dev_history.groupby('developer').agg(
    dev_n_players=('steamid', 'nunique'),
    dev_n_games_played=('appid', 'nunique'),
    dev_avg_playtime=('playtime_forever', 'mean'),
    dev_avg_target=('target', 'mean'),
).reset_index()

# Item-level dev features (max/mean across devs of a game)
item_dev = dev_exploded.merge(dev_stats, on='developer', how='left')
item_dev_agg = item_dev.groupby('appid').agg(
    item_dev_max_players=('dev_n_players', 'max'),
    item_dev_mean_players=('dev_n_players', 'mean'),
    item_dev_max_games=('dev_n_games_played', 'max'),
    item_dev_avg_target=('dev_avg_target', 'mean'),
).reset_index().fillna(0)

# --- Publisher exploded ---
pub_exploded = games[['appid', 'publishers_list']].explode('publishers_list')
pub_exploded = pub_exploded.rename(columns={'publishers_list': 'publisher'})
pub_exploded = pub_exploded.dropna(subset=['publisher'])

pub_history = pub_exploded.merge(
    history[['appid', 'steamid', 'playtime_forever', 'target']], on='appid', how='inner'
)
pub_stats = pub_history.groupby('publisher').agg(
    pub_n_players=('steamid', 'nunique'),
    pub_avg_target=('target', 'mean'),
).reset_index()

item_pub = pub_exploded.merge(pub_stats, on='publisher', how='left')
item_pub_agg = item_pub.groupby('appid').agg(
    item_pub_max_players=('pub_n_players', 'max'),
    item_pub_avg_target=('pub_avg_target', 'mean'),
).reset_index().fillna(0)

print(f"  Unique developers: {len(dev_stats)}, publishers: {len(pub_stats)}")

# --- User-Developer affinity ---
print("\n[B2] User-Developer affinity...")

user_dev = dev_history.groupby(['steamid', 'developer']).agg(
    user_dev_n_games=('appid', 'nunique'),
    user_dev_total_pt=('playtime_forever', 'sum'),
).reset_index()

# Для каждого appid — список developers
dev_per_appid = dev_exploded.groupby('appid')['developer'].apply(list).reset_index()

# User-Publisher affinity
user_pub = pub_history.groupby(['steamid', 'publisher']).agg(
    user_pub_n_games=('appid', 'nunique'),
).reset_index()

pub_per_appid = pub_exploded.groupby('appid')['publisher'].apply(list).reset_index()

print(f"  User-dev pairs: {len(user_dev)}, User-pub pairs: {len(user_pub)}")

# ============================================================
# C. ENHANCED USER FEATURES
# ============================================================
print("\n[C] Enhanced user features...")

# C1. Richer playtime stats
user_stats = history.groupby('steamid').agg(
    user_total_games=('appid', 'count'),
    user_total_playtime=('playtime_forever', 'sum'),
    user_avg_playtime=('playtime_forever', 'mean'),
    user_median_playtime=('playtime_forever', 'median'),
    user_std_playtime=('playtime_forever', 'std'),
    user_max_playtime=('playtime_forever', 'max'),
    user_avg_target_hist=('target', 'mean'),
    user_std_target_hist=('target', 'std'),
    user_n_high_target=('target', lambda x: (x >= 4).sum()),
    user_n_low_target=('target', lambda x: (x <= 2).sum()),
    user_pct_zero_playtime=('playtime_forever', lambda x: (x == 0).mean()),
).reset_index()

user_stats['user_std_playtime'] = user_stats['user_std_playtime'].fillna(0)
user_stats['user_std_target_hist'] = user_stats['user_std_target_hist'].fillna(0)
user_stats['user_total_playtime_log'] = np.log1p(user_stats['user_total_playtime'])
user_stats['user_high_target_ratio'] = user_stats['user_n_high_target'] / user_stats['user_total_games'].clip(lower=1)

# C2. Platform preferences
user_plat = history.groupby('steamid').agg(
    pt_windows=('playtime_windows_forever', 'sum'),
    pt_mac=('playtime_mac_forever', 'sum'),
    pt_linux=('playtime_linux_forever', 'sum'),
    pt_deck=('playtime_deck_forever', 'sum'),
    pt_total=('playtime_forever', 'sum'),
).reset_index()

for p in ['windows', 'mac', 'linux', 'deck']:
    user_plat[f'user_pct_{p}'] = user_plat[f'pt_{p}'] / user_plat['pt_total'].clip(lower=1)

user_plat = user_plat[['steamid', 'user_pct_windows', 'user_pct_mac', 'user_pct_linux', 'user_pct_deck']]

# C3. Free game preference
hist_free = history[['steamid', 'appid']].merge(games[['appid', 'is_free']], on='appid', how='left')
user_free = hist_free.groupby('steamid')['is_free'].agg(
    user_free_ratio='mean'
).reset_index()
user_free['user_free_ratio'] = user_free['user_free_ratio'].fillna(0.5)

# C4. User profile features
user_profile = users[['steamid', 'loccountrycode', 'timecreated']].copy()
user_profile['loccountrycode'] = user_profile['loccountrycode'].fillna('UNKNOWN')

# Account age (относительно данных)
max_time = history['rtime_last_played'].max()
if max_time == 0:
    max_time = 1776010854  # примерное значение из interactions
user_profile['account_age_days'] = (max_time - user_profile['timecreated'].fillna(max_time)) / 86400
user_profile['account_age_days'] = user_profile['account_age_days'].clip(lower=0)
user_profile = user_profile.drop(columns=['timecreated'])

# C5. Playtime-WEIGHTED genre/category affinity (ключевое улучшение!)
print("\n[C5] Playtime-weighted affinity...")

hist_mh = history[['steamid', 'appid', 'playtime_forever']].merge(
    item_feat_mh[['appid'] + mh_cols], on='appid', how='inner'
)
hist_mh['log_pt'] = np.log1p(hist_mh['playtime_forever'])

# Count-based affinity (оригинальная)
user_game_counts = hist_mh.groupby('steamid').size().reset_index(name='_cnt')
user_aff_count = hist_mh.groupby('steamid')[mh_cols].sum().reset_index()
user_aff_count = user_aff_count.merge(user_game_counts, on='steamid')
for col in mh_cols:
    user_aff_count[f'uaff_{col}'] = (user_aff_count[col] / user_aff_count['_cnt']).astype('float32')
user_aff_count = user_aff_count[['steamid'] + [f'uaff_{c}' for c in mh_cols]]

# Playtime-weighted affinity
for col in mh_cols:
    hist_mh[f'w_{col}'] = hist_mh[col] * hist_mh['log_pt']

w_cols = [f'w_{c}' for c in mh_cols]
user_aff_wt = hist_mh.groupby('steamid')[w_cols].sum().reset_index()
user_total_wt = hist_mh.groupby('steamid')['log_pt'].sum().reset_index()
user_total_wt.columns = ['steamid', '_total_w']
user_aff_wt = user_aff_wt.merge(user_total_wt, on='steamid')

for col in mh_cols:
    user_aff_wt[f'uwaff_{col}'] = (user_aff_wt[f'w_{col}'] / user_aff_wt['_total_w'].clip(lower=0.001)).astype('float32')
user_aff_wt = user_aff_wt[['steamid'] + [f'uwaff_{c}' for c in mh_cols]]

# C6. Country frequency encoding (из history)
country_counts = user_profile['loccountrycode'].value_counts()
user_profile['country_freq'] = user_profile['loccountrycode'].map(country_counts).fillna(0).astype(int)

print("All base features computed!")

# ============================================================
# SAVE
# ============================================================
item_pop.to_parquet('artifacts/v2_item_pop.parquet', index=False)
item_dev_agg.to_parquet('artifacts/v2_item_dev.parquet', index=False)
item_pub_agg.to_parquet('artifacts/v2_item_pub.parquet', index=False)
user_stats.to_parquet('artifacts/v2_user_stats.parquet', index=False)
user_plat.to_parquet('artifacts/v2_user_plat.parquet', index=False)
user_free.to_parquet('artifacts/v2_user_free.parquet', index=False)
user_profile.to_parquet('artifacts/v2_user_profile.parquet', index=False)
user_aff_count.to_parquet('artifacts/v2_user_aff_count.parquet', index=False)
user_aff_wt.to_parquet('artifacts/v2_user_aff_wt.parquet', index=False)
user_dev.to_parquet('artifacts/v2_user_dev.parquet', index=False)
user_pub.to_parquet('artifacts/v2_user_pub.parquet', index=False)
dev_per_appid.to_parquet('artifacts/v2_dev_per_appid.parquet', index=False)
pub_per_appid.to_parquet('artifacts/v2_pub_per_appid.parquet', index=False)

print("\nAll artifacts saved!")


Genre columns: 20, Category columns: 20

[A] Item popularity features...
  Items with popularity data: 18953

[B] Developer/Publisher features...
  Unique developers: 11377, publishers: 8265

[B2] User-Developer affinity...
  User-dev pairs: 264494, User-pub pairs: 213993

[C] Enhanced user features...


/tmp/ipykernel_6361/2976125991.py:173: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  user_free['user_free_ratio'] = user_free['user_free_ratio'].fillna(0.5)



[C5] Playtime-weighted affinity...
All base features computed!

All artifacts saved!


In [4]:
import pandas as pd
import numpy as np

# ============================================================
# ЗАГРУЗКА АРТЕФАКТОВ
# ============================================================
item_feat_mh = pd.read_parquet('artifacts/item_features_multihot.parquet')
item_pop = pd.read_parquet('artifacts/v2_item_pop.parquet')
item_dev_agg = pd.read_parquet('artifacts/v2_item_dev.parquet')
item_pub_agg = pd.read_parquet('artifacts/v2_item_pub.parquet')

user_stats = pd.read_parquet('artifacts/v2_user_stats.parquet')
user_plat = pd.read_parquet('artifacts/v2_user_plat.parquet')
user_free = pd.read_parquet('artifacts/v2_user_free.parquet')
user_profile = pd.read_parquet('artifacts/v2_user_profile.parquet')
user_aff_count = pd.read_parquet('artifacts/v2_user_aff_count.parquet')
user_aff_wt = pd.read_parquet('artifacts/v2_user_aff_wt.parquet')

user_dev = pd.read_parquet('artifacts/v2_user_dev.parquet')
user_pub = pd.read_parquet('artifacts/v2_user_pub.parquet')
dev_per_appid = pd.read_parquet('artifacts/v2_dev_per_appid.parquet')
pub_per_appid = pd.read_parquet('artifacts/v2_pub_per_appid.parquet')

games = pd.read_csv('data/game_details.csv')

mh_genre_cols = [c for c in item_feat_mh.columns if c.startswith('genres_')]
mh_cat_cols = [c for c in item_feat_mh.columns if c.startswith('categories_')]
mh_cols = mh_genre_cols + mh_cat_cols

print("All artifacts loaded!")


def compute_dev_pub_affinity(df, dev_per_appid, pub_per_appid, user_dev, user_pub):
    """Compute developer/publisher affinity cross-features"""
    print("  Computing developer affinity...")
    
    # Developer affinity
    df_dev = df[['steamid', 'appid']].merge(dev_per_appid, on='appid', how='left')
    df_dev['developer'] = df_dev['developer'].apply(lambda x: x if isinstance(x, list) else [])
    df_dev = df_dev.explode('developer')
    
    df_dev = df_dev.merge(user_dev, on=['steamid', 'developer'], how='left')
    df_dev['user_dev_n_games'] = df_dev['user_dev_n_games'].fillna(0)
    df_dev['user_dev_total_pt'] = df_dev['user_dev_total_pt'].fillna(0)
    
    dev_agg = df_dev.groupby(['steamid', 'appid']).agg(
        user_dev_max_games=('user_dev_n_games', 'max'),
        user_dev_sum_games=('user_dev_n_games', 'sum'),
        user_dev_max_pt=('user_dev_total_pt', 'max'),
        user_dev_has_history=('user_dev_n_games', lambda x: (x > 0).any().astype(int)),
    ).reset_index()
    
    df = df.merge(dev_agg, on=['steamid', 'appid'], how='left')
    for col in ['user_dev_max_games', 'user_dev_sum_games', 'user_dev_max_pt', 'user_dev_has_history']:
        df[col] = df[col].fillna(0)
    df['user_dev_max_pt_log'] = np.log1p(df['user_dev_max_pt'])
    
    # Publisher affinity
    print("  Computing publisher affinity...")
    df_pub = df[['steamid', 'appid']].merge(pub_per_appid, on='appid', how='left')
    df_pub['publisher'] = df_pub['publisher'].apply(lambda x: x if isinstance(x, list) else [])
    df_pub = df_pub.explode('publisher')
    
    df_pub = df_pub.merge(user_pub, on=['steamid', 'publisher'], how='left')
    df_pub['user_pub_n_games'] = df_pub['user_pub_n_games'].fillna(0)
    
    pub_agg = df_pub.groupby(['steamid', 'appid']).agg(
        user_pub_max_games=('user_pub_n_games', 'max'),
        user_pub_sum_games=('user_pub_n_games', 'sum'),
        user_pub_has_history=('user_pub_n_games', lambda x: (x > 0).any().astype(int)),
    ).reset_index()
    
    df = df.merge(pub_agg, on=['steamid', 'appid'], how='left')
    for col in ['user_pub_max_games', 'user_pub_sum_games', 'user_pub_has_history']:
        df[col] = df[col].fillna(0)
    
    return df


def assemble_dataset_v2(base_file_path):
    print(f"\nAssembling: {base_file_path}")
    df = pd.read_parquet(base_file_path)
    df = df.rename(columns={'score': 'als_score', 'rank': 'als_rank'})
    original_len = len(df)
    
    # ============================================================
    # 1. ALS TRANSFORMATIONS
    # ============================================================
    print("  [1] ALS transformations...")
    df['als_score_log'] = np.log1p(df['als_score'])
    df['als_rank_norm'] = df['als_rank'] / 300.0
    df['als_rank_inv'] = 1.0 / df['als_rank']
    df['als_rank_inv_sqrt'] = 1.0 / np.sqrt(df['als_rank'])
    
    # Per-user ALS normalization
    als_stats = df.groupby('steamid')['als_score'].agg(['mean', 'std', 'max', 'min']).reset_index()
    als_stats.columns = ['steamid', 'als_umean', 'als_ustd', 'als_umax', 'als_umin']
    df = df.merge(als_stats, on='steamid')
    df['als_score_zscore'] = (df['als_score'] - df['als_umean']) / df['als_ustd'].clip(lower=0.001)
    df['als_score_minmax'] = (df['als_score'] - df['als_umin']) / (df['als_umax'] - df['als_umin']).clip(lower=0.001)
    df = df.drop(columns=['als_umean', 'als_ustd', 'als_umax', 'als_umin'])
    
    # ============================================================
    # 2. USER FEATURES
    # ============================================================
    print("  [2] User features...")
    df = df.merge(user_profile, on='steamid', how='left')
    df = df.merge(user_stats, on='steamid', how='left')
    df = df.merge(user_plat, on='steamid', how='left')
    df = df.merge(user_free, on='steamid', how='left')
    
    # ============================================================
    # 3. ITEM FEATURES
    # ============================================================
    print("  [3] Item features...")
    df = df.merge(item_feat_mh, on='appid', how='left')
    df = df.merge(item_pop, on='appid', how='left')
    df = df.merge(item_dev_agg, on='appid', how='left')
    df = df.merge(item_pub_agg, on='appid', how='left')
    
    # Game type
    game_type = games[['appid', 'type']].copy()
    game_type['type'] = game_type['type'].fillna('unknown')
    df = df.merge(game_type, on='appid', how='left')
    df['type'] = df['type'].fillna('unknown')
    
    # ============================================================
    # 4. CROSS-FEATURES: MATCH SCORES (самое важное!)
    # ============================================================
    print("  [4] Match scores...")
    
    # Genre match (count-based affinity × item genre)
    genre_match = np.zeros(len(df), dtype='float32')
    genre_wmatch = np.zeros(len(df), dtype='float32')
    user_genre_norm_sq = np.zeros(len(df), dtype='float32')
    item_genre_norm_sq = np.zeros(len(df), dtype='float32')
    user_wgenre_norm_sq = np.zeros(len(df), dtype='float32')

    # Подготовим user affinity
    df = df.merge(user_aff_count, on='steamid', how='left')
    df = df.merge(user_aff_wt, on='steamid', how='left')
    
    for col in mh_genre_cols:
        u_col = f'uaff_{col}'
        uw_col = f'uwaff_{col}'
        i_val = df[col].fillna(0).values.astype('float32')
        u_val = df[u_col].fillna(0).values.astype('float32') if u_col in df.columns else np.zeros(len(df), dtype='float32')
        uw_val = df[uw_col].fillna(0).values.astype('float32') if uw_col in df.columns else np.zeros(len(df), dtype='float32')
        
        genre_match += u_val * i_val
        genre_wmatch += uw_val * i_val
        user_genre_norm_sq += u_val ** 2
        item_genre_norm_sq += i_val ** 2
        user_wgenre_norm_sq += uw_val ** 2
    
    df['genre_match_dot'] = genre_match
    df['genre_wmatch_dot'] = genre_wmatch
    
    # Cosine similarity for genres
    denom = np.sqrt(user_genre_norm_sq) * np.sqrt(item_genre_norm_sq)
    df['genre_match_cosine'] = np.where(denom > 0, genre_match / denom, 0).astype('float32')
    
    denom_w = np.sqrt(user_wgenre_norm_sq) * np.sqrt(item_genre_norm_sq)
    df['genre_wmatch_cosine'] = np.where(denom_w > 0, genre_wmatch / denom_w, 0).astype('float32')
    
    # Category match
    cat_match = np.zeros(len(df), dtype='float32')
    cat_wmatch = np.zeros(len(df), dtype='float32')
    user_cat_norm_sq = np.zeros(len(df), dtype='float32')
    item_cat_norm_sq = np.zeros(len(df), dtype='float32')
    
    for col in mh_cat_cols:
        u_col = f'uaff_{col}'
        uw_col = f'uwaff_{col}'
        i_val = df[col].fillna(0).values.astype('float32')
        u_val = df[u_col].fillna(0).values.astype('float32') if u_col in df.columns else np.zeros(len(df), dtype='float32')
        uw_val = df[uw_col].fillna(0).values.astype('float32') if uw_col in df.columns else np.zeros(len(df), dtype='float32')
        
        cat_match += u_val * i_val
        cat_wmatch += uw_val * i_val
        user_cat_norm_sq += u_val ** 2
        item_cat_norm_sq += i_val ** 2
    
    df['cat_match_dot'] = cat_match
    df['cat_wmatch_dot'] = cat_wmatch
    
    denom_c = np.sqrt(user_cat_norm_sq) * np.sqrt(item_cat_norm_sq)
    df['cat_match_cosine'] = np.where(denom_c > 0, cat_match / denom_c, 0).astype('float32')
    
    # Total match
    df['total_match_dot'] = df['genre_match_dot'] + df['cat_match_dot']
    df['total_wmatch_dot'] = df['genre_wmatch_dot'] + df['cat_wmatch_dot']
    df['total_match_cosine'] = (df['genre_match_cosine'] + df['cat_match_cosine']) / 2
    
    # ============================================================
    # 5. DEVELOPER / PUBLISHER AFFINITY CROSS-FEATURES
    # ============================================================
    print("  [5] Developer/Publisher affinity...")
    df = compute_dev_pub_affinity(df, dev_per_appid, pub_per_appid, user_dev, user_pub)
    
    # ============================================================
    # 6. ДОПОЛНИТЕЛЬНЫЕ CROSS-FEATURES
    # ============================================================
    print("  [6] Additional cross-features...")
    
    # Platform match: user plays on Linux, game supports Linux
    df['platform_match_mac'] = (df['user_pct_mac'].fillna(0) > 0.1).astype(int) * df.get('platforms_count', pd.Series(0)).fillna(0)
    df['platform_match_linux'] = (df['user_pct_linux'].fillna(0) > 0.1).astype(int)
    
    # Free match: user prefers free games × item is free
    df['free_match'] = df['user_free_ratio'].fillna(0.5) * df['is_free'].fillna(0)
    
    # ALS × item popularity interaction
    df['als_x_item_pop'] = df['als_score'] * df['item_n_players_log'].fillna(0)
    df['als_x_recommendations'] = df['als_score'] * df['recommendations_log'].fillna(0)
    
    # ALS × match score interactions
    df['als_x_genre_match'] = df['als_score'] * df['genre_match_cosine']
    df['als_x_total_match'] = df['als_score'] * df['total_match_cosine']
    
    # ALS × dev affinity
    df['als_x_dev_history'] = df['als_score'] * df['user_dev_has_history']
    
    # User engagement × item quality
    df['user_engagement_x_item_target'] = df['user_avg_target_hist'].fillna(3) * df['item_avg_target_hist'].fillna(3)
    
    # Item popularity relative to user library size
    df['item_pop_vs_user_lib'] = df['item_n_players'].fillna(0) / df['user_total_games'].clip(lower=1)
    
    # ============================================================
    # 7. CLEANUP
    # ============================================================
    print("  [7] Cleanup...")
    
    # Drop individual affinity columns (kept match scores)
    uaff_cols = [c for c in df.columns if c.startswith('uaff_') or c.startswith('uwaff_')]
    df = df.drop(columns=uaff_cols)
    
    # Fill NAs
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    df[numeric_cols] = df[numeric_cols].fillna(0)
    
    assert len(df) == original_len, f"Row count changed! {original_len} -> {len(df)}"
    print(f"  Final shape: {df.shape}")
    
    return df


# ============================================================
# ASSEMBLE TRAIN & TEST
# ============================================================
train_v2 = assemble_dataset_v2('artifacts/train_reranker_base.parquet')
test_v2 = assemble_dataset_v2('artifacts/test_reranker_base.parquet')

train_v2 = train_v2.sort_values('steamid')
test_v2 = test_v2.sort_values('steamid')

train_v2.to_parquet('artifacts/train_v2.parquet', index=False)
test_v2.to_parquet('artifacts/test_v2.parquet', index=False)

print(f"\n{'='*60}")
print(f"Train: {train_v2.shape}")
print(f"Test:  {test_v2.shape}")
print(f"\nAll columns ({len(train_v2.columns)}):")
for i, col in enumerate(train_v2.columns):
    print(f"  {i:3d}. {col}")


All artifacts loaded!

Assembling: artifacts/train_reranker_base.parquet
  [1] ALS transformations...
  [2] User features...
  [3] Item features...
  [4] Match scores...


/tmp/ipykernel_6361/3358889399.py:161: RuntimeWarning: invalid value encountered in divide
  df['genre_match_cosine'] = np.where(denom > 0, genre_match / denom, 0).astype('float32')
/tmp/ipykernel_6361/3358889399.py:164: RuntimeWarning: invalid value encountered in divide
  df['genre_wmatch_cosine'] = np.where(denom_w > 0, genre_wmatch / denom_w, 0).astype('float32')
/tmp/ipykernel_6361/3358889399.py:188: RuntimeWarning: invalid value encountered in divide
  df['cat_match_cosine'] = np.where(denom_c > 0, cat_match / denom_c, 0).astype('float32')


  [5] Developer/Publisher affinity...
  Computing developer affinity...
  Computing publisher affinity...
  [6] Additional cross-features...
  [7] Cleanup...
  Final shape: (1064400, 123)

Assembling: artifacts/test_reranker_base.parquet
  [1] ALS transformations...
  [2] User features...
  [3] Item features...
  [4] Match scores...
  [5] Developer/Publisher affinity...
  Computing developer affinity...


/tmp/ipykernel_6361/3358889399.py:161: RuntimeWarning: invalid value encountered in divide
  df['genre_match_cosine'] = np.where(denom > 0, genre_match / denom, 0).astype('float32')
/tmp/ipykernel_6361/3358889399.py:164: RuntimeWarning: invalid value encountered in divide
  df['genre_wmatch_cosine'] = np.where(denom_w > 0, genre_wmatch / denom_w, 0).astype('float32')
/tmp/ipykernel_6361/3358889399.py:188: RuntimeWarning: invalid value encountered in divide
  df['cat_match_cosine'] = np.where(denom_c > 0, cat_match / denom_c, 0).astype('float32')


  Computing publisher affinity...
  [6] Additional cross-features...
  [7] Cleanup...
  Final shape: (266100, 123)

Train: (1064400, 123)
Test:  (266100, 123)

All columns (123):
    0. steamid
    1. appid
    2. als_score
    3. als_rank
    4. target
    5. als_score_log
    6. als_rank_norm
    7. als_rank_inv
    8. als_rank_inv_sqrt
    9. als_score_zscore
   10. als_score_minmax
   11. loccountrycode
   12. account_age_days
   13. country_freq
   14. user_total_games
   15. user_total_playtime
   16. user_avg_playtime
   17. user_median_playtime
   18. user_std_playtime
   19. user_max_playtime
   20. user_avg_target_hist
   21. user_std_target_hist
   22. user_n_high_target
   23. user_n_low_target
   24. user_pct_zero_playtime
   25. user_total_playtime_log
   26. user_high_target_ratio
   27. user_pct_windows
   28. user_pct_mac
   29. user_pct_linux
   30. user_pct_deck
   31. user_free_ratio
   32. is_free
   33. recommendations_log
   34. age_years
   35. platforms_count
 

In [5]:
import pandas as pd
import numpy as np
from catboost import CatBoostRanker, Pool
from catboost.utils import get_gpu_device_count
import mlflow
import mlflow.catboost

print(f"GPU: {get_gpu_device_count()}")

train = pd.read_parquet("artifacts/train_v2.parquet").sort_values("steamid")
test = pd.read_parquet("artifacts/test_v2.parquet").sort_values("steamid")

# Определяем колонки
drop_cols = ["steamid", "appid", "target"]
cat_features = ["loccountrycode", "type"]

# Убедимся что cat features — строки
for cf in cat_features:
    train[cf] = train[cf].fillna("unknown").astype(str)
    test[cf] = test[cf].fillna("unknown").astype(str)

feature_cols = [c for c in train.columns if c not in drop_cols]
print(f"Features: {len(feature_cols)}")
print(f"Cat features: {cat_features}")

X_train = train[feature_cols]
y_train = train["target"]
q_train = train["steamid"]

X_test = test[feature_cols]
y_test = test["target"]
q_test = test["steamid"]

train_pool = Pool(data=X_train, label=y_train, group_id=q_train, cat_features=cat_features)
test_pool = Pool(data=X_test, label=y_test, group_id=q_test, cat_features=cat_features)

# ============================================================
# TRAIN
# ============================================================
mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("Steam_RecSys_Reranking_v2")

with mlflow.start_run(run_name="CatBoost_v2_rich_features"):
    params = {
        "iterations": 2000,
        "learning_rate": 0.05,
        "depth": 6,
        "l2_leaf_reg": 3.0,
        "loss_function": "PairLogitPairwise",
        "custom_metric": ["NDCG:top=10", "MAP:top=10"],
        "eval_metric": "NDCG:top=10",
        "early_stopping_rounds": 100,
        "random_seed": 42,
        "task_type": "GPU",
        "devices": "0",
        "border_count": 128,
        "bootstrap_type": "Bayesian",
        "verbose": 100,
    }

    mlflow.log_params(params)
    mlflow.log_param("n_features", len(feature_cols))

    model = CatBoostRanker(**params)
    model.fit(train_pool, eval_set=test_pool, verbose=100)

    best_score = model.get_best_score()
    best_iteration = model.get_best_iteration()
    best_ndcg_10 = best_score["validation"]["NDCG:top=10;type=Base"]
    
    try:
        best_map_10 = best_score["validation"]["MAP:top=10;type=Base"]
        mlflow.log_metric("best_map_10", best_map_10)
        print(f"Best MAP@10:  {best_map_10:.4f}")
    except:
        pass

    mlflow.log_metric("best_iteration", best_iteration)
    mlflow.log_metric("best_ndcg_10", best_ndcg_10)

    model.save_model("artifacts/catboost_ranker_v2.cbm")
    mlflow.catboost.log_model(model, artifact_path="model_v2")

    print(f"\nBest iteration: {best_iteration}")
    print(f"Best NDCG@10:  {best_ndcg_10:.4f}")

    # Feature importance
    fi = model.get_feature_importance(train_pool)
    fi_df = pd.DataFrame({
        'feature': feature_cols,
        'importance': fi
    }).sort_values('importance', ascending=False)
    
    print(f"\nTop-30 features:")
    print(fi_df.head(30).to_string(index=False))
    
    fi_df.to_csv('artifacts/feature_importance_v2.csv', index=False)


GPU: 1
Features: 120
Cat features: ['loccountrycode', 'type']


2026/05/15 22:10:37 INFO mlflow.tracking.fluent: Experiment with name 'Steam_RecSys_Reranking_v2' does not exist. Creating a new experiment.


Groupwise loss function. OneHotMaxSize set to 10


Default metric period is 5 because MAP, NDCG is/are not implemented for GPU
Metric NDCG:top=10;type=Base is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time
Metric NDCG:top=10;type=Base is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time
Metric MAP:top=10 is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time


0:	test: 0.2470349	best: 0.2470349 (0)	total: 124ms	remaining: 4m 7s
100:	test: 0.3480213	best: 0.3485431 (98)	total: 4.19s	remaining: 1m 18s
200:	test: 0.3666014	best: 0.3668513 (194)	total: 8.15s	remaining: 1m 12s
300:	test: 0.3719202	best: 0.3720854 (296)	total: 12.1s	remaining: 1m 8s
400:	test: 0.3753035	best: 0.3757873 (352)	total: 16s	remaining: 1m 3s
500:	test: 0.3782972	best: 0.3790719 (469)	total: 20s	remaining: 59.9s
600:	test: 0.3782285	best: 0.3803521 (569)	total: 24s	remaining: 55.8s


2026/05/15 22:11:06 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


bestTest = 0.3803520671
bestIteration = 569
Shrink model to first 570 iterations.

Best iteration: 569
Best NDCG@10:  0.3804

Top-30 features:
                       feature  importance
                     age_years    0.034912
          item_pct_high_target    0.012439
        item_pct_zero_playtime    0.010363
                item_n_players    0.010169
          item_avg_target_hist    0.004304
              account_age_days    0.003725
          item_median_playtime    0.002689
              als_score_minmax    0.002630
           recommendations_log    0.002464
          item_std_target_hist    0.002448
                     als_score    0.002370
            item_n_players_log    0.001728
                 als_score_log    0.001570
             item_std_playtime    0.001370
               user_free_ratio    0.001342
          item_pub_max_players    0.001326
              cat_match_cosine    0.000927
           item_total_playtime    0.000849
             user_n_low_target    0.0008